## 영화 리뷰 감정 분석기

### 출력 파서: `EnumOutputParser` + `Enum` 클래스

In [22]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [23]:
from langchain_openai import ChatOpenAI

PROVIDER = "groq"

LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini", 
    },
}
MAX_RETRIES = 5  


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed, 
        max_retries=MAX_RETRIES,
    )

llm = get_llm(temperature=0.7) 

In [24]:
from enum import Enum
from langchain_classic.output_parsers import EnumOutputParser   # langchain 1.x: classic 패키지

class Sentiment(Enum):
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "보통"     

parser = EnumOutputParser(enum=Sentiment)

format_instructions = parser.get_format_instructions()
print(format_instructions)                

print(repr(parser.parse(" 보통\n")))        

Select one of the following options: 긍정, 부정, 보통
<Sentiment.NEUTRAL: '보통'>


In [25]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 영화 리뷰 감정 분석 전문가입니다.\n"
     "리뷰를 읽고 감정을 하나로 분류하세요.\n"
     "- 긍정: 칭찬, 만족, 추천\n"
     "- 부정: 불만, 실망, 비추천\n"
     "- 보통: 장단점이 섞였거나 무난하다는 평가\n"
     "설명 없이 값 하나만 답하세요.\n\n"
     "{format_instructions}"),
    ("user", "리뷰: {review}"),
]).partial(format_instructions=format_instructions)

llm = get_llm(temperature=0)

chain = prompt | llm | parser

In [26]:
from langchain_core.exceptions import OutputParserException

reviews = [
    "이 영화 정말 재미없어요. 시간 낭비였습니다.",
    "배우들의 연기가 훌륭하고 스토리도 감동적이었어요!",
    "그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.",
    "OST는 좋았는데 결말이 너무 허무했어요.",
    "올해 본 영화 중 최고! 두 번 봤습니다.",
]

EMOJI = {Sentiment.POSITIVE: "[★★★★★]", Sentiment.NEGATIVE: "[★☆☆☆☆]", Sentiment.NEUTRAL: "[★★★☆☆]"}

print(f"{'감정':<6} | 리뷰")
print("-" * 60)
for review in reviews:
    try:
        result = chain.invoke({"review": review})
        print(f"{EMOJI[result]} {result.value:<4} | {review}")
    except OutputParserException as e:
        print(f"실패  | {review}  ({str(e).splitlines()[0]})")

감정     | 리뷰
------------------------------------------------------------
[★☆☆☆☆] 부정   | 이 영화 정말 재미없어요. 시간 낭비였습니다.
[★★★★★] 긍정   | 배우들의 연기가 훌륭하고 스토리도 감동적이었어요!
[★★★☆☆] 보통   | 그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.
[★★★☆☆] 보통   | OST는 좋았는데 결말이 너무 허무했어요.
[★★★★★] 긍정   | 올해 본 영화 중 최고! 두 번 봤습니다.


In [27]:
from collections import Counter

results = chain.batch([{"review": r} for r in reviews])
counts = Counter(results)

for s in Sentiment:
    print(f"{s.value}: {counts.get(s, 0)}건")

긍정: 2건
부정: 1건
보통: 2건


In [28]:
from langchain_classic.output_parsers import OutputFixingParser
from langchain_core.prompts import PromptTemplate

fix_prompt = PromptTemplate.from_template(
    """아래 Completion은 감정 분류 결과인데 허용된 형식이 아닙니다.
Completion의 **의미를 그대로 유지**하여 허용된 값 중 하나로 변환하세요.
(예: Positive → 긍정, Negative → 부정, 부정적입니다 → 부정)

허용 형식: {instructions}
Completion: {completion}
오류: {error}

값 하나만 출력하세요:"""
)

fixing_parser = OutputFixingParser.from_llm(
    llm=llm, parser=parser, prompt=fix_prompt, max_retries=1   # ← prompt 지정
)

bad_outputs = ["Positive", "부정적입니다", "보통인 것 같아요."]

for bad in bad_outputs:
    try:
        parser.parse(bad)
        print(f"'{bad}' → 기본 파서 성공")
    except OutputParserException:
        fixed = fixing_parser.parse(bad)
        print(f"'{bad}' → 기본 파서 실패 → 복구: {fixed.value}")

fixing_chain = prompt | llm | fixing_parser
print(fixing_chain.invoke({"review": reviews[0]}).value)

'Positive' → 기본 파서 실패 → 복구: 긍정
'부정적입니다' → 기본 파서 실패 → 복구: 부정
'보통인 것 같아요.' → 기본 파서 실패 → 복구: 보통
부정
